In [9]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [10]:
class LSTMNet(nn.Module):
    def __init__(self, input_size=12288, hidden_size=64, num_layers=1, speed_size=1, output_size=1):
        super(LSTMNet, self).__init__()
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True).to(self.device)
        self.fc = nn.Linear(hidden_size + speed_size, output_size).to(self.device)
        self.relu = nn.ReLU().to(self.device)
        self.tanh = nn.Tanh().to(self.device)

    def forward(self, image, speed):
        image = image.to(self.device)
        speed = speed.to(self.device)

        # Initialize hidden state with zeros
        h0 = torch.zeros(self.num_layers, 1, self.hidden_size).to(self.device)
        c0 = torch.zeros(self.num_layers, 1, self.hidden_size).to(self.device)

        # Forward propagate LSTM
        out, _ = self.lstm(image.unsqueeze(0), (h0, c0))

        # Take the output from the last time step
        out = out[:, -1, :]

        # Concatenate with speed
        out = torch.cat((out, speed), dim=1)

        # Fully connected layer
        out = self.fc(out)

        # Activation function
        out = self.tanh(out)

        return out

In [11]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel.csv')

In [17]:
batch_size = min(len(X), len(S), len(y))
X_tensor = torch.tensor(X.values[:batch_size], dtype=torch.float32).to(device='cuda')
S_tensor = torch.tensor(S.values[:batch_size], dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values[:batch_size, 0], dtype=torch.float32).to(device='cuda')

# Reshape the input data for the LSTM
X_tensor = X_tensor.reshape(batch_size, 1, -1).to(device='cuda')

# Create the dataset
dataset = TensorDataset(X_tensor, S_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

In [18]:
print(len(X_tensor))
print(len(S_tensor))
print(len(y_tensor))

13118
13118
13118


In [19]:
print(X_tensor[0])
print(X_tensor[0].size())

tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
torch.Size([1, 12288])


In [20]:
print(S_tensor[0])
print(S_tensor[0].size())

tensor([104.], device='cuda:0')
torch.Size([1])


In [21]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor(0.0296, device='cuda:0')
torch.Size([])


In [ ]:
# tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
# torch.Size([1, 12288])

# tensor([[117]], device='cuda:0')
# torch.Size([1, 1])

# tensor([[-0.2253]], device='cuda:0', grad_fn=<TanhBackward0>)
# torch.Size([1, 1])

In [9]:
model = LSTMNet().cuda()

# Define your loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [10]:
num_epochs = 150
# Assuming you have your training data loaded into train_loader
for epoch in range(num_epochs):
    for images, speeds, labels in train_loader:
        images = images.to('cuda')
        speeds = speeds.to('cuda')
        labels = labels.to('cuda')

        # Forward pass
        outputs = model(images, speeds)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/150], Loss: 0.8629
Epoch [2/150], Loss: 0.8023
Epoch [3/150], Loss: 0.7339
Epoch [4/150], Loss: 0.6803
Epoch [5/150], Loss: 0.6638
Epoch [6/150], Loss: 0.6613
Epoch [7/150], Loss: 0.6531
Epoch [8/150], Loss: 0.6389
Epoch [9/150], Loss: 0.6226
Epoch [10/150], Loss: 0.6064
Epoch [11/150], Loss: 0.5909
Epoch [12/150], Loss: 0.5760
Epoch [13/150], Loss: 0.5612
Epoch [14/150], Loss: 0.5460
Epoch [15/150], Loss: 0.5304
Epoch [16/150], Loss: 0.5150
Epoch [17/150], Loss: 0.5005
Epoch [18/150], Loss: 0.4877
Epoch [19/150], Loss: 0.4761
Epoch [20/150], Loss: 0.4651
Epoch [21/150], Loss: 0.4538
Epoch [22/150], Loss: 0.4418
Epoch [23/150], Loss: 0.4293
Epoch [24/150], Loss: 0.4166
Epoch [25/150], Loss: 0.4040
Epoch [26/150], Loss: 0.3911
Epoch [27/150], Loss: 0.3777
Epoch [28/150], Loss: 0.3636
Epoch [29/150], Loss: 0.3489
Epoch [30/150], Loss: 0.3336
Epoch [31/150], Loss: 0.3181
Epoch [32/150], Loss: 0.3022
Epoch [33/150], Loss: 0.2860
Epoch [34/150], Loss: 0.2692
Epoch [35/150], Loss: 0

In [11]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_lstm.pth')

In [12]:
print(X_tensor.size(0))
print(S_tensor.size(0))
print(y_tensor.size(0))

13118
13118
13118
